# Portfolio API — Binance (`slug: binance`)

Exercises all `/portfolio/*` endpoints scoped to the `binance` source.
US equity holdings via CDP browser fetch — attaches to the authenticated
Binance web app and intercepts the holdings XHR.

**Auth pre-req:** Log in to [binance.com](https://binance.com) inside the AlphaForge Anton Chrome session (`--remote-debugging-port=9299`). Set `BINANCE_USER_ID` in `backend/.env.cred.local`.

In [ ]:
import json, os
from pathlib import Path

SLUG = "binance"
MODE = "http"          # "in_process" | "http"
BASE = "http://localhost:8000/api/v1"

AF_USERNAME = os.getenv("AF_USERNAME", "admin")
AF_PASSWORD = os.getenv("AF_PASSWORD", "alphaforge-anton-dev")

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""


def _login() -> str:
    r = client.post(
        f"{PREFIX}/auth/token",
        data={"username": AF_USERNAME, "password": AF_PASSWORD},
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Auth failed ({r.status_code}): {r.text}. "
            "Set AF_USERNAME / AF_PASSWORD env vars if you changed admin creds."
        )
    return r.json()["access_token"]


def _ensure_auth() -> None:
    if "Authorization" not in client.headers:
        client.headers["Authorization"] = f"Bearer {_login()}"


def _request(method: str, path: str, **kw):
    _ensure_auth()
    r = client.request(method, f"{PREFIX}{path}", **kw)
    if r.status_code == 401:
        client.headers["Authorization"] = f"Bearer {_login()}"
        r = client.request(method, f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text


def get(path, **kw):  return _request("GET", path, **kw)
def post(path, **kw): return _request("POST", path, **kw)

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

_ensure_auth()
print(f"Mode: {MODE}  slug: {SLUG}  authed as: {AF_USERNAME}")

## 1. Source info

`status: ready` when `BINANCE_USER_ID` is set in `.env.cred.local`, `unconfigured` otherwise.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Attaches to Chrome via CDP, navigates to `binance.com/dashboard`, and intercepts
the crypto wallet holdings XHR. Result is cached to
`~/.alphaforge-anton/portfolio-dumps/binance-holdings-live.csv`.

> Requires `MODE="http"` with a live server and an open Chrome session
> where you are already logged in to binance.com.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:5]:
    sym = "$" if h.get("currency") == "USD" else "₹"
    print(f"  {h['symbol']:14}  qty={h['quantity']:<10}  avg={sym}{h['avg_price']:>10,.2f}  ltp={sym}{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Holdings — binance only

US equity holdings (Binance's crypto portfolio).

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    sym = "$" if h.get("currency") == "USD" else "₹"
    print(f"  {h['symbol']:14}  qty={h['quantity']:<10}  avg={sym}{h['avg_price']:>10,.2f}  ltp={sym}{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 4. Allocation (binance)

Expected: `equity`-only (crypto) allocation.

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation (USD):")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ${a['value']:>14,.2f}  ({a['pct']:>5.1f}%)")

## 5. Treemap (binance)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 6. Rebalance (binance)

In [ ]:
status, body = get("/portfolio/rebalance", params={"source": SLUG})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 7. Standalone dump (bypass FastAPI)

Directly runs the CDP fetch + CSV write without starting the server.
Useful for testing auth and CSV output end-to-end.

## 7. Free cash (USD)

Binance holds crypto, so cash is in **USD**.

`GET /portfolio/cash` — cached snapshot, always instant.  
`POST /portfolio/cash/{slug}/sync` — opens the crypto wallet holdings page via CDP and reads `cash_available_for_trade` from `/account/basic` (~15-25 s). Requires an active Chrome CDP session.

In [ ]:
# Cached snapshot — instant, no CDP round-trip
_, snap = get("/portfolio/cash")
entry = next((c for c in snap.get("cash", []) if c["source"] == SLUG), None)
if entry:
    sym = "$" if entry.get("currency") == "USD" else "₹"
    avail = "✓" if entry["cash_available"] else "✗ (not yet synced)"
    print(f"Cached  [{avail}] {sym}{entry.get('cash', 0):,.2f}  as_of={entry.get('cash_as_of') or 'never'}")
else:
    print("Source not found in /cash response")

# Live sync via CDP
print("\nSyncing …")
status, body = post(f"/portfolio/cash/{SLUG}/sync")
print(f"Status: {status}")
if status == 200:
    c = body["cash"]
    sym = "$" if c.get("currency") == "USD" else "₹"
    print(f"Fresh   [✓] {sym}{c.get('cash', 0):>12,.2f}  as_of={c.get('cash_as_of')}")
else:
    pp(body)

In [ ]:
import asyncio, sys
sys.path.insert(0, str(Path.cwd().parent))  # add backend/ to path

from app.modules.brokers.binance.binance_dump import dump_binance

path = await dump_binance()
print(f"Dumped → {path}")

## 8. Reset binance cache

In [ ]:
if MODE == "in_process":
    from app.modules.brokers.registry import SOURCES
    SOURCES[SLUG].reset()
    status, body = get(f"/portfolio/sources/{SLUG}")
    print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")
else:
    print("Switch MODE to 'in_process' to reset the in-memory cache directly.")
    print("Or restart the server to clear all cached holdings.")